# 6D. Per-Exercise Density TCN Sweep

This notebook is the first explicit temporal-counting follow-up after the frozen scalar TCN baselines. It reuses the current Stage 5 pose-sequence dataset but changes the prediction target from a single whole-video count to a temporal density curve whose sum gives the final count.


## Reason, Approach, Result Interpretation

**Reason**
- The scalar TCN baseline in `6_ALL` and the per-exercise temporal tuning in `6B` showed that the pipeline has real signal, but absolute errors remain high.
- The manual keypoint-weighting experiment in `6C` mostly failed, which suggests the next bottleneck is the temporal counting formulation rather than another small feature tweak.

**Approach**
- Keep the same Stage 5 normalized pose sequences and the strongest `seq_len` per exercise from `6B`.
- Replace direct whole-video count regression with a density-based TCN that predicts a non-negative temporal curve.
- Train the density curve with a weak temporal target: evenly spaced Gaussian peaks whose total mass matches the true count.

**Result interpretation**
- If the density model lowers `MAE` or raises `Within-1` beyond the best `6B` run, then making repetition structure explicit is helping.
- If the density model stays flat or worse, the next bottleneck is likely representation quality rather than scalar-vs-density supervision alone.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup

**Why this section exists**
- The density experiment needs the new density trainer, the existing scalar trainer it imports, and the baseline-comparison script in the same Drive project.

**Approach**
- Mount Drive.
- Sync the current repo copies of the scalar trainer, density trainer, and comparison script into the Drive project.
- Resolve the full `pose_sequence_index.csv` that already includes squat.

**How to interpret the result**
- If the paths print correctly and `SEQUENCE_INDEX exists = True`, the environment is ready.
- If the index is missing, rerun Stage 5 before starting this notebook.


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

SCALAR_TRAINER_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
DENSITY_TRAINER_REL = Path('artifacts/3_Modeling/train_pose_count_density_tcn.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')

SCALAR_TRAINER_SRC = CODE_ROOT / SCALAR_TRAINER_REL
SCALAR_TRAINER_DST = DRIVE_PROJECT_ROOT / SCALAR_TRAINER_REL
DENSITY_TRAINER_SRC = CODE_ROOT / DENSITY_TRAINER_REL
DENSITY_TRAINER_DST = DRIVE_PROJECT_ROOT / DENSITY_TRAINER_REL
COMPARE_SRC = CODE_ROOT / COMPARE_REL
COMPARE_DST = DRIVE_PROJECT_ROOT / COMPARE_REL

for src, dst in [
    (SCALAR_TRAINER_SRC, SCALAR_TRAINER_DST),
    (DENSITY_TRAINER_SRC, DENSITY_TRAINER_DST),
    (COMPARE_SRC, COMPARE_DST),
]:
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
SEQUENCE_INDEX = ANNOTATION_DIR / 'pose_sequence_index.csv'

print('SCALAR_TRAINER_DST =', SCALAR_TRAINER_DST)
print('DENSITY_TRAINER_DST =', DENSITY_TRAINER_DST)
print('COMPARE_DST =', COMPARE_DST)
print('SEQUENCE_INDEX =', SEQUENCE_INDEX)
print('SEQUENCE_INDEX exists =', SEQUENCE_INDEX.exists())


## Dataset Coverage Check

**Why this section exists**
- This experiment reuses the strongest per-exercise `seq_len` settings from `6B`, so every target exercise needs nonzero `train` and `valid` rows.

**Approach**
- Read `pose_sequence_index.csv`.
- Display the split counts for the selected exercises.

**How to interpret the result**
- If a target exercise is missing, the issue is still upstream in Stage 5 rather than in the density TCN itself.


In [ ]:
import pandas as pd

TARGET_EXERCISES = ['bench_pressing', 'pommelhorse', 'pull_up', 'push_up', 'squat']
seq_df = pd.read_csv(SEQUENCE_INDEX)
counts_df = seq_df.groupby(['type', 'split']).size().unstack(fill_value=0).sort_index()
display(counts_df.loc[TARGET_EXERCISES])


## Density Experiment Design

**Why this section exists**
- The next step should isolate the temporal formulation, not reopen the sequence-length search.

**Approach**
- Reuse the best `seq_len` discovered in `6B` for each exercise.
- Compare each new density run directly against its best scalar `6B` reference.
- Use weak temporal supervision from pseudo density targets rather than requiring new annotations.

**How to interpret the result**
- If the density runs beat the `6B` scalar baselines, explicit temporal counting is helping.
- If they do not, the project has evidence that the main limitation lies elsewhere.


In [ ]:
import pandas as pd

EXERCISE_CONFIGS = [
    {
        'exercise': 'bench_pressing',
        'seq_len': 192,
        'baseline_run': 'pose_count_tcn_bench_pressing_seq192',
        'density_run': 'pose_count_density_tcn_bench_pressing_seq192',
    },
    {
        'exercise': 'pommelhorse',
        'seq_len': 192,
        'baseline_run': 'pose_count_tcn_pommelhorse_seq192',
        'density_run': 'pose_count_density_tcn_pommelhorse_seq192',
    },
    {
        'exercise': 'pull_up',
        'seq_len': 192,
        'baseline_run': 'pose_count_tcn_pull_up_seq192',
        'density_run': 'pose_count_density_tcn_pull_up_seq192',
    },
    {
        'exercise': 'push_up',
        'seq_len': 128,
        'baseline_run': 'pose_count_tcn_push_up_seq128',
        'density_run': 'pose_count_density_tcn_push_up_seq128',
    },
    {
        'exercise': 'squat',
        'seq_len': 256,
        'baseline_run': 'pose_count_tcn_squat_seq256',
        'density_run': 'pose_count_density_tcn_squat_seq256',
    },
]

EPOCHS = 80
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
CHANNELS = 96
KERNEL_SIZE = 3
NUM_BLOCKS = 4
DROPOUT = 0.2
PATIENCE = 15
LOSS = 'l1'
DENSITY_LOSS = 'mse'
DENSITY_LOSS_WEIGHT = 0.5
SMOOTHNESS_WEIGHT = 0.01
PSEUDO_SIGMA_SCALE = 0.35
EVAL_TRANSFORM = 'raw'
SELECTION_METRIC = 'mae'
SAMPLER = 'balanced_count'
TIME_WARP_RANGE = 0.12
FEATURE_NOISE_STD = 0.02
FRAME_DROPOUT_PROB = 0.03

display(pd.DataFrame(EXERCISE_CONFIGS))
print('DENSITY_LOSS =', DENSITY_LOSS)
print('DENSITY_LOSS_WEIGHT =', DENSITY_LOSS_WEIGHT)
print('SMOOTHNESS_WEIGHT =', SMOOTHNESS_WEIGHT)
print('PSEUDO_SIGMA_SCALE =', PSEUDO_SIGMA_SCALE)


## Training Execution

**Why this section exists**
- This is the actual density-model training stage.

**Approach**
- Reuse the current pose-sequence dataset.
- Train one density TCN per exercise using the strongest `seq_len` from `6B`.
- Keep failures non-fatal so one exercise does not block the whole experiment.

**How to interpret the result**
- A clean sweep means every configured density run finished and wrote a new artifact folder.
- If one run fails, debug it separately without discarding the completed runs.


In [ ]:
import subprocess
import pandas as pd

training_failures = []
for cfg in EXERCISE_CONFIGS:
    cmd = [
        'python', str(DENSITY_TRAINER_DST),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--index-csv', str(SEQUENCE_INDEX),
        '--run-name', cfg['density_run'],
        '--exercise', cfg['exercise'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--lr', str(LR),
        '--weight-decay', str(WEIGHT_DECAY),
        '--channels', str(CHANNELS),
        '--kernel-size', str(KERNEL_SIZE),
        '--num-blocks', str(NUM_BLOCKS),
        '--dropout', str(DROPOUT),
        '--patience', str(PATIENCE),
        '--loss', LOSS,
        '--density-loss', DENSITY_LOSS,
        '--density-loss-weight', str(DENSITY_LOSS_WEIGHT),
        '--smoothness-weight', str(SMOOTHNESS_WEIGHT),
        '--pseudo-sigma-scale', str(PSEUDO_SIGMA_SCALE),
        '--eval-transform', EVAL_TRANSFORM,
        '--selection-metric', SELECTION_METRIC,
        '--sampler', SAMPLER,
        '--time-warp-range', str(TIME_WARP_RANGE),
        '--feature-noise-std', str(FEATURE_NOISE_STD),
        '--frame-dropout-prob', str(FRAME_DROPOUT_PROB),
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        training_failures.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'density_run': cfg['density_run'],
            'returncode': exc.returncode,
        })
        print(f"FAILED: {cfg['exercise']} (returncode={exc.returncode})")

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All density runs completed.')


## Raw Metric Review

**Why this section exists**
- The first check should compare the direct validation metrics of the new density runs against the best scalar `6B` runs.

**Approach**
- Load `metrics_summary.json` for both the scalar `6B` reference and the new density run.
- Compare `valid_mae`, `valid_rmse`, and `valid_within_1` per exercise.

**How to interpret the result**
- Lower `valid_mae` and higher `valid_within_1` in the density row mean the explicit temporal formulation is helping.
- If the density row is flat or worse, the next limitation is likely not fixed by density supervision alone.


In [ ]:
import json
import pandas as pd

rows = []
for cfg in EXERCISE_CONFIGS:
    for variant, run_name in [('baseline_6B', cfg['baseline_run']), ('density_tcn', cfg['density_run'])]:
        metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'metrics_summary.json'
        if not metrics_path.exists():
            continue
        with open(metrics_path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'best_epoch': metrics.get('best_epoch'),
            'valid_mae': metrics['valid_metrics']['mae'],
            'valid_rmse': metrics['valid_metrics']['rmse'],
            'valid_within_1': metrics['valid_metrics']['within_1'],
        })

raw_df = pd.DataFrame(rows)
if raw_df.empty:
    print('No metrics_summary.json files found yet.')
else:
    display(raw_df.sort_values(['exercise', 'variant']))
    pivot_df = raw_df.pivot(index=['exercise', 'seq_len'], columns='variant', values=['valid_mae', 'valid_within_1'])
    pivot_df.columns = ['_'.join(col).strip() for col in pivot_df.columns.values]
    pivot_df = pivot_df.reset_index()
    if 'valid_mae_baseline_6B' in pivot_df.columns and 'valid_mae_density_tcn' in pivot_df.columns:
        pivot_df['delta_mae_density_minus_baseline6B'] = pivot_df['valid_mae_density_tcn'] - pivot_df['valid_mae_baseline_6B']
    if 'valid_within_1_baseline_6B' in pivot_df.columns and 'valid_within_1_density_tcn' in pivot_df.columns:
        pivot_df['delta_within_1_density_minus_baseline6B'] = pivot_df['valid_within_1_density_tcn'] - pivot_df['valid_within_1_baseline_6B']
    display(pivot_df.sort_values('exercise'))


## Baseline-Comparison Review

**Why this section exists**
- Lower raw `MAE` is useful, but we also need to know whether the density model improves value over the trivial train-split baseline and over the best scalar `6B` reference.

**Approach**
- Ensure each density run has a `baseline_comparison_summary.json` file.
- Load the summary for the scalar `6B` run and the density run.
- Compare both their trivial-baseline deltas and their direct density-vs-scalar gap.

**How to interpret the result**
- More negative `delta_mae_vs_trivial` is better.
- More positive `delta_within_1_vs_trivial` is better.
- Negative `delta_mae_density_minus_baseline6B` and positive `delta_within_1_density_minus_baseline6B` indicate that explicit temporal counting beat the best scalar TCN for that exercise.


In [ ]:
import subprocess
import json
import pandas as pd

comparison_failures = []
for cfg in EXERCISE_CONFIGS:
    run_dir = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['density_run']
    pred_path = run_dir / 'predictions.csv'
    if not pred_path.exists():
        continue
    summary_path = run_dir / 'baseline_comparison_summary.json'
    if summary_path.exists():
        continue
    cmd = [
        'python', str(COMPARE_DST),
        '--index-csv', str(SEQUENCE_INDEX),
        '--predictions-csv', str(pred_path),
        '--exercise', cfg['exercise'],
    ]
    print('\nComparing:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        comparison_failures.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'density_run': cfg['density_run'],
            'returncode': exc.returncode,
        })

rows = []
for cfg in EXERCISE_CONFIGS:
    for variant, run_name in [('baseline_6B', cfg['baseline_run']), ('density_tcn', cfg['density_run'])]:
        summary_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'baseline_comparison_summary.json'
        if not summary_path.exists():
            continue
        with open(summary_path, 'r', encoding='utf-8') as f:
            summary = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'model_mae': summary['model_metrics']['mae'],
            'baseline_mae': summary['baseline_metrics']['mae'],
            'delta_mae_vs_trivial': summary['delta_vs_baseline']['mae'],
            'model_within_1': summary['model_metrics']['within_1'],
            'baseline_within_1': summary['baseline_metrics']['within_1'],
            'delta_within_1_vs_trivial': summary['delta_vs_baseline']['within_1'],
            'model_beats_baseline_rows': summary['row_level']['model_beats_baseline'],
            'valid_rows': summary['row_level']['valid_rows'],
        })

compare_df = pd.DataFrame(rows)
if not compare_df.empty:
    display(compare_df.sort_values(['exercise', 'variant']))
    compare_pivot = compare_df.pivot(index=['exercise', 'seq_len'], columns='variant', values=['model_mae', 'model_within_1', 'delta_mae_vs_trivial', 'delta_within_1_vs_trivial'])
    compare_pivot.columns = ['_'.join(col).strip() for col in compare_pivot.columns.values]
    compare_pivot = compare_pivot.reset_index()
    if 'model_mae_baseline_6B' in compare_pivot.columns and 'model_mae_density_tcn' in compare_pivot.columns:
        compare_pivot['delta_mae_density_minus_baseline6B'] = compare_pivot['model_mae_density_tcn'] - compare_pivot['model_mae_baseline_6B']
    if 'model_within_1_baseline_6B' in compare_pivot.columns and 'model_within_1_density_tcn' in compare_pivot.columns:
        compare_pivot['delta_within_1_density_minus_baseline6B'] = compare_pivot['model_within_1_density_tcn'] - compare_pivot['model_within_1_baseline_6B']
    display(compare_pivot.sort_values('exercise'))
else:
    print('No baseline comparison summaries found yet.')

if comparison_failures:
    display(pd.DataFrame(comparison_failures))


## Density Artifact Check

**Why this section exists**
- The density formulation should produce a temporal signal you can inspect, not just a scalar count.

**Approach**
- Load the `valid_density_curves.npz` artifact for one finished run.
- Preview the stored arrays and plot a few density curves if plotting is available.

**How to interpret the result**
- Curves with visible peaks indicate the model is using temporal structure.
- Very flat curves suggest the density formulation is not adding much beyond a reparameterized scalar regression.


In [ ]:
import numpy as np

RUN_TO_INSPECT = EXERCISE_CONFIGS[-1]['density_run']
artifact_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / RUN_TO_INSPECT / 'valid_density_curves.npz'
print('artifact_path =', artifact_path)

if artifact_path.exists():
    artifact = np.load(artifact_path, allow_pickle=True)
    print('keys =', artifact.files)
    print('pred_density shape =', artifact['pred_density'].shape)
    print('target_density shape =', artifact['target_density'].shape)
    try:
        import matplotlib.pyplot as plt
        n_show = min(3, artifact['pred_density'].shape[0])
        for idx in range(n_show):
            plt.figure(figsize=(10, 3))
            plt.plot(artifact['target_density'][idx], label='pseudo_target', linewidth=2)
            plt.plot(artifact['pred_density'][idx], label='pred_density', linewidth=2)
            plt.title(f"{artifact['names'][idx]} | true={artifact['true_counts'][idx]:.1f} | pred={artifact['pred_counts'][idx]:.2f}")
            plt.legend()
            plt.show()
    except Exception as exc:
        print('Plotting skipped:', exc)
else:
    print('Density artifact not found yet.')
